# 01 Data: conversion, audit, exploration and the snapshot panel

This notebook turns the six raw CSV files into tables ready for analysis, checks that they can be trusted, runs the descriptive checks behind the research design and builds the panel of snapshots used by every later notebook. A snapshot is one user on one day the user visited the feed.

Inputs: `csv_data/` with the six raw CSV files, unchanged. Outputs: `parquet/` with one table per CSV, the impressions with the feed position on one scale and `panel.parquet`, and `outputs/01_*`, `02_*` and `03_*`.

The comments between cells describe the run on all users. With `--sample 10` the numbers differ.

## 1. Setup

In [1]:
# Environment setup: libraries, project paths and shared settings
import csv, json, time, hashlib
import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.stats import chi2_contingency
from sklearn.metrics import roc_auc_score, average_precision_score

from thesis.config import RAW, P, O, connect, SAMPLE, TAG
from thesis.click_model import K, card_prior_sql, platform_rate

print(f"Raw data: {RAW.name}/   Parquet: {P.name}/   Outputs: {O.name}/   Sample: {SAMPLE}% of users")

Raw data: csv_data/   Parquet: parquet/   Outputs: outputs/   Sample: 100% of users


## 2. Conversion to Parquet

The impression file cannot be read with a standard CSV parser, because its first field, the list of cards a user swiped through after a click, contains commas and is not always quoted. The other twelve fields never contain commas, so each line is split from the right into thirteen fields, and any line that does not split this way is written to a reject file. The conversion can be interrupted and resumed.

In [2]:
# Conversion functions: the impression file (split from the right) and the five small tables
OUT = P
BATCH = 1_000_000
MAX_ROWS = 25_000_000
STATE = P / "_conversion_state.json"
REJECTS = P / "_conversion_rejects.txt"

EXPECTED_HEADER = ["detailMlogInfoList", "dt", "impressPosition", "impressTime", "isClick",
                   "isComment", "isIntoPersonalHomepage", "isShare", "isViewComment", "isLike",
                   "mlogId", "mlogViewTime", "userId"]

MAIN_SCHEMA = pa.schema([
    ("row_id", pa.int64()), ("userId", pa.string()), ("mlogId", pa.string()),
    ("dt", pa.int16()), ("impressPosition", pa.int32()), ("impressTime", pa.int64()),
    ("isClick", pa.int8()), ("isLike", pa.int8()), ("isComment", pa.int8()),
    ("isShare", pa.int8()), ("isViewComment", pa.int8()), ("isIntoPersonalHomepage", pa.int8()),
    ("mlogViewTime", pa.float64()), ("has_detail", pa.bool_()),
    ("detail_quoted", pa.bool_()), ("detail_n", pa.int32())])
DETAIL_SCHEMA = pa.schema([("row_id", pa.int64()), ("detail_raw", pa.string())])


def to_float(s):
    """'NA' and empty strings become missing values."""
    return None if s in ("NA", "") else float(s)


def convert_impressions():
    """One pass of at most MAX_ROWS rows. Returns True when the whole file has been converted."""
    if STATE.exists() and json.loads(STATE.read_text())["done"]:
        return True

    src = RAW / "impression_data.csv"
    with open(src, "r", encoding="utf-8") as f:
        header = f.readline().rstrip("\n").split(",")
        header_end = f.tell()
    assert header == EXPECTED_HEADER, header

    if STATE.exists():
        state = json.loads(STATE.read_text())
    else:
        state = dict(offset=header_end, part=0, rows=0, rejects=0, detail_rows=0,
                     detail_unquoted_with_commas=0, detail_empty_list=0, done=False)

    (OUT / "impressions").mkdir(exist_ok=True)
    (OUT / "impression_detail").mkdir(exist_ok=True)
    part = state["part"]
    w_main = pq.ParquetWriter(OUT / "impressions" / f"part-{part:03d}.parquet", MAIN_SCHEMA, compression="zstd")
    w_det = pq.ParquetWriter(OUT / "impression_detail" / f"part-{part:03d}.parquet", DETAIL_SCHEMA, compression="zstd")
    rejects = open(REJECTS, "a")
    cols = {name: [] for name in MAIN_SCHEMA.names}
    det = {"row_id": [], "detail_raw": []}
    t0, n_this = time.time(), 0

    def flush():
        w_main.write_table(pa.Table.from_pydict(cols, schema=MAIN_SCHEMA))
        for values in cols.values():
            values.clear()
        if det["row_id"]:
            w_det.write_table(pa.Table.from_pydict(det, schema=DETAIL_SCHEMA))
            det["row_id"].clear()
            det["detail_raw"].clear()

    with open(src, "rb") as fb:
        fb.seek(state["offset"])
        while n_this < MAX_ROWS:
            bline = fb.readline()
            if not bline:
                state["done"] = True
                break
            line = bline.decode("utf-8").rstrip("\n")
            parts = line.rsplit(",", 12)                     # the last 12 fields never contain commas
            if len(parts) != 13:
                state["rejects"] += 1
                rejects.write(line[:500] + "\n")
                continue

            rid = state["rows"]
            state["rows"] += 1
            n_this += 1

            # swipe-down list: strip the quotes if present, count the cards in it
            d = parts[0]
            quoted = d.startswith('"') and d.endswith('"')
            raw = d[1:-1].replace('""', '"') if quoted else d
            has = raw not in ("NA", "")
            n = None
            if has:
                state["detail_rows"] += 1
                if not quoted and "," in raw:
                    state["detail_unquoted_with_commas"] += 1
                n = raw.count("'mlogId'")
                if raw.strip() == "[]":
                    state["detail_empty_list"] += 1
                det["row_id"].append(rid)
                det["detail_raw"].append(raw)

            cols["row_id"].append(rid)
            cols["userId"].append(parts[12])
            cols["mlogId"].append(parts[10])
            cols["dt"].append(int(parts[1]))
            cols["impressPosition"].append(int(parts[2]))
            cols["impressTime"].append(int(parts[3]))
            cols["isClick"].append(int(parts[4]))
            cols["isComment"].append(int(parts[5]))
            cols["isIntoPersonalHomepage"].append(int(parts[6]))
            cols["isShare"].append(int(parts[7]))
            cols["isViewComment"].append(int(parts[8]))
            cols["isLike"].append(int(parts[9]))
            cols["mlogViewTime"].append(to_float(parts[11]))
            cols["has_detail"].append(has)
            cols["detail_quoted"].append(quoted)
            cols["detail_n"].append(n)
            if len(cols["row_id"]) >= BATCH:
                flush()
        state["offset"] = fb.tell()

    flush()
    w_main.close()
    w_det.close()
    rejects.close()
    state["part"] = part + 1
    STATE.write_text(json.dumps(state, indent=2))
    print(("DONE " if state["done"] else "PARTIAL ") + json.dumps(state), f"{time.time() - t0:.0f}s", flush=True)
    return state["done"]


def convert_small():
    """The five small tables, read as text so that 'NA' strings and IDs are kept exactly."""
    con = duckdb.connect()
    for name in ["user_demographics", "mlog_demographics", "mlog_stats", "creator_demographics", "creator_stats"]:
        con.execute(f"""COPY (SELECT * FROM read_csv('{RAW / (name + '.csv')}', header=true, all_varchar=true,
                       nullstr='', quote='"'))
                       TO '{OUT / (name + '.parquet')}' (FORMAT parquet, COMPRESSION zstd)""")
        n = con.execute(f"SELECT count(*) FROM '{OUT / (name + '.parquet')}'").fetchone()[0]
        print(name, n, flush=True)

In [3]:
# Run the conversion (skipped when the state file says it is complete)
if not (OUT / "creator_stats.parquet").exists():
    convert_small()
while not convert_impressions():
    pass
pd.Series(json.loads(STATE.read_text()), name="conversion").to_frame()

,conversion
offset,6343915917
part,3
rows,57750395
rejects,0
detail_rows,733733
detail_unquoted_with_commas,0
detail_empty_list,6537
done,True


All 57,750,395 impression rows split into thirteen fields, so no line was rejected.

## 3. Keys, links and consistency

The tables must join correctly and the recorded actions must be consistent before anything is analyzed. The checks cover row counts and unique keys, duplicate impressions, the day field against the timestamp in Beijing time, actions recorded without the click that should precede them, and users or cards missing from one table.

In [4]:
# Audit queries on the Parquet tables
con = connect(keep_order=False)
con.execute(f"CREATE VIEW imp AS SELECT * FROM read_parquet('{P}/impressions/*.parquet')")
for t in ["user_demographics", "mlog_demographics", "mlog_stats", "creator_demographics", "creator_stats"]:
    con.execute(f"CREATE VIEW {t} AS SELECT * FROM '{P}/{t}.parquet'")


def q(sql):
    return con.execute(sql).fetchone()


out = {}
out["impressions_rows"] = q("SELECT count(*) FROM imp")[0]
out["impressions_users"] = q("SELECT count(DISTINCT userId) FROM imp")[0]
out["impressions_cards"] = q("SELECT count(DISTINCT mlogId) FROM imp")[0]
out["dt_min_max"] = q("SELECT min(dt), max(dt) FROM imp")
out["dup_user_card_time"] = q("SELECT count(*) - count(DISTINCT (userId, mlogId, impressTime)) FROM imp")[0]
out["position_min_max"] = q("SELECT min(impressPosition), max(impressPosition) FROM imp")
out["clicks"] = q("SELECT sum(isClick) FROM imp")[0]
out["viewtime_nonnull_without_click"] = q("SELECT count(*) FROM imp WHERE isClick=0 AND mlogViewTime IS NOT NULL")[0]
out["click_without_viewtime"] = q("SELECT count(*) FROM imp WHERE isClick=1 AND mlogViewTime IS NULL")[0]
out["actions_without_click"] = dict(zip(["like", "comment", "share", "viewcomment", "homepage"], q("""
  SELECT sum(isLike), sum(isComment), sum(isShare), sum(isViewComment), sum(isIntoPersonalHomepage)
  FROM imp WHERE isClick=0""")))
out["detail_on_nonclick"] = q("SELECT count(*) FROM imp WHERE has_detail AND isClick=0")[0]
out["dt_vs_beijing_day_mismatch"] = q("""SELECT count(*) FROM imp
  WHERE dt <> day(to_timestamp(impressTime/1000) AT TIME ZONE 'Asia/Shanghai')""")[0]
out["user_demo_rows_unique"] = q("SELECT count(*), count(DISTINCT userId) FROM user_demographics")
out["imp_users_not_in_demo"] = q("""SELECT count(*) FROM (SELECT DISTINCT userId FROM imp) i
  ANTI JOIN user_demographics u USING (userId)""")[0]
out["imp_cards_not_in_catalog"] = q("""SELECT count(*) FROM (SELECT DISTINCT mlogId FROM imp) i
  ANTI JOIN mlog_demographics m USING (mlogId)""")[0]
out["catalog_cards_not_in_imp"] = q("""SELECT count(*) FROM mlog_demographics m
  ANTI JOIN (SELECT DISTINCT mlogId FROM imp) i USING (mlogId)""")[0]
out["tenure_ge_100_months"] = q("""SELECT count(*) FROM user_demographics
  WHERE registeredMonthCnt NOT IN ('NA','') AND CAST(registeredMonthCnt AS INT) >= 100""")[0]
out["new_users_tenure_le1"] = q("""SELECT count(*) FROM user_demographics
  WHERE registeredMonthCnt NOT IN ('NA','') AND CAST(registeredMonthCnt AS INT) <= 1""")[0]
con.close()

(O / "01_audit.json").write_text(json.dumps(out, indent=2, default=str))
pd.Series({k: str(v) for k, v in out.items()}, name="value").to_frame()

,value
impressions_rows,57750395
impressions_users,2085602
impressions_cards,211066
dt_min_max,"(1, 30)"
dup_user_card_time,0
position_min_max,"(0, 14924)"
clicks,2870438
viewtime_nonnull_without_click,8886
click_without_viewtime,92248
actions_without_click,"{'like': 840, 'comment': 123, 'share': 514, 'v..."


The tables join cleanly. There are 57,750,395 impressions of 211,066 cards to 2,085,602 users, no impression appears twice, and the day field matches the timestamp on every row.

The problems are small. 69 users have no profile and 7 cards have no catalog entry. Each action that should follow a click was recorded without one about 1,100 times at most. Viewing time is missing for 3.2% of clicks. Thirteen accounts report an impossible age, and 41,896 catalog cards were never shown to a sampled user. No row needs to be dropped.

### Feed position on one scale

The documentation numbers the four cards of the first page 1 to 4. The impressions mix three numberings: pages numbered 0 to 3, pages numbered 1 to 4, and pages on which two copies of the first page are logged at the same moment, each numbered 1 to 4. The next cell rewrites the impressions with each card's place counted from 0 on every page, and every later step reads this table.

In [5]:
# Feed position on one scale: rewrite the impressions with each card's place counted from 0 (see thesis.positions)
import os
from thesis import positions
if os.environ.get("THESIS_FORCE", "0") == "1" or not (P / positions.OUT / "_done").exists():
    con = connect(keep_order=False)
    position_fix = positions.build(con, P, O)
    con.close()
else:
    position_fix = json.loads((O / "01_position_fix.json").read_text())
position_fix


{'impressions_by_offset_source': {'user': 2073324,
  'session': 22566422,
  'top': 33096035,
  'default': 14614},
 'impressions': 57750395,
 'impressions_raw': 57750395,
 'top_pages_by_scheme': {'from_0': 3112850,
  'doubled': 1178053,
  'from_1': 3772733}}

## 4. Missing age and gender

Age and gender are the platform's own estimates and are missing for about 36% of users. If they were missing completely at random, the share missing would be the same for every kind of account. The cells below compare it across account ages and user levels.

In [6]:
# Missing age and gender by account level and by account age (all users in the user table)
con = duckdb.connect()
users = con.execute(f"""SELECT TRY_CAST(age AS DOUBLE) AS age, gender,
    TRY_CAST(registeredMonthCnt AS INT) AS tenure, TRY_CAST(TRY_CAST(level AS DOUBLE) AS INT) AS level
  FROM '{P}/user_demographics.parquet'""").df()
con.close()

users["missing"] = users.age.isna()
TENURE_BINS = [-1, 1, 3, 6, 12, 24, 36, 60, 1000]
TENURE_LABELS = ["0-1", "2-3", "4-6", "7-12", "13-24", "25-36", "37-60", "61+"]
users["tenure_months"] = pd.cut(users.tenure, TENURE_BINS, labels=TENURE_LABELS)

checks = {
    "users": len(users),
    "age_missing_share": round(float(users.missing.mean()), 4),
    "age_and_gender_missing_together_share": round(float((users.age.isna() == users.gender.isna()).mean()), 4),
    "gender_unknown_category": int((users.gender == "unknown").sum()),
}
by_level = users.groupby("level").missing.agg(users="size", missing_share="mean")
by_tenure = users.groupby("tenure_months", observed=True).missing.agg(users="size", missing_share="mean")
by_both = users.pivot_table(index="level", columns="tenure_months", values="missing", aggfunc="mean", observed=True)

for name, table in (("level", by_level), ("tenure_months", by_tenure)):
    chi2, p, dof, _ = chi2_contingency(pd.crosstab(users[name], users.missing))
    checks[f"chi2_by_{name}"] = {"chi2": round(float(chi2), 1), "dof": int(dof), "p_value": float(p)}

(O / "01_profile_missingness.json").write_text(json.dumps(checks, indent=2))
pd.concat({"by level": by_level, "by account age (months)": by_tenure}).round(3).to_csv(O / "01_profile_missingness.csv")
pd.Series({k: str(v) for k, v in checks.items()}, name="value").to_frame()

,value
users,2085533
age_missing_share,0.3555
age_and_gender_missing_together_share,1.0
gender_unknown_category,30319
chi2_by_level,"{'chi2': 148993.2, 'dof': 10, 'p_value': 0.0}"
chi2_by_tenure_months,"{'chi2': 258491.9, 'dof': 7, 'p_value': 0.0}"


In [7]:
# Share missing by level (rows) and account age in months (columns)
(by_both * 100).round(1)

tenure_months,0-1,2-3,4-6,7-12,13-24,25-36,37-60,61+
level,,,,,,,,
0,99.8,64.6,7.2,14.6,33.9,49.4,54.6,60.0
1,99.1,61.0,5.7,9.5,20.4,31.1,34.7,56.2
2,98.0,56.6,5.6,7.7,15.1,22.6,25.7,33.3
3,97.3,53.4,6.8,8.4,12.0,17.7,20.1,33.6
4,97.5,54.6,10.0,10.7,12.3,16.3,21.2,28.9
5,58.1,49.3,15.0,15.3,14.1,15.9,17.9,19.6
6,37.8,30.2,32.6,29.9,23.6,21.7,21.0,24.6
7,51.0,49.1,53.8,54.4,43.7,36.4,29.9,27.8
8,85.7,53.8,68.8,72.5,64.4,54.3,43.4,37.4


In [8]:
pd.concat({"by level": by_level, "by account age (months)": by_tenure}).round(3)

users  missing_share
by level                0       62455          0.702
                        1       93050          0.497
                        2       73904          0.369
                        3      125506          0.296
                        4      129858          0.287
                        5      261372          0.190
                        6      431683          0.243
                        7      542419          0.378
                        8      297250          0.498
                        9       63312          0.614
                        10       4315          0.701
by account age (months) 0-1    116843          0.978
                        2-3     95112          0.547
                        4-6    116922          0.153
                        7-12   246305          0.242
                        13-24  503275          0.287
                        25-36  576932          0.340
                        37-60  405644          0.363
                        61+     24091          0.387

The share missing differs sharply across account ages and user levels (chi-square tests, p < 0.001). Accounts of one month or less almost never have an estimate (98% missing). The share is lowest at four to six months (15%) and rises again to 39% for the oldest accounts. By user level it is 70% at level 0, 19% at level 5 and 70% at level 10.

Having an estimate therefore depends on the kind of account, so age and gender are used only to describe users, never as model inputs.

## 5. Exposure and clicking

Users first seen on days 1 to 10 are followed through their first week and then over the two weeks after it. The question is whether clicking at least once in those two weeks measures interest or mostly how many cards the user was shown.

In [9]:
# User-relative windows: first week in the log and the 14 days after it
OBS, OUT_LEN, LAST_D0 = 7, 14, 10

con = connect(keep_order=False)
con.execute(f"CREATE VIEW imp AS SELECT userId, dt, isClick FROM read_parquet('{P}/impressions_place/d*.parquet')")
con.execute(f"CREATE VIEW users AS SELECT * FROM '{P}/user_demographics.parquet'")

# users by first day seen, with how many hold an account at most one month old on 1 December (new accounts) and how
# many have no account age recorded
first_day = con.execute("""SELECT d0, count(*) AS users,
    sum((TRY_CAST(u.registeredMonthCnt AS INT) <= 1)::INT) AS users_new_account,
    sum((TRY_CAST(u.registeredMonthCnt AS INT) IS NULL)::INT) AS users_account_age_missing
  FROM (SELECT userId, min(dt) AS d0 FROM imp GROUP BY userId) f LEFT JOIN users u USING (userId)
  GROUP BY d0 ORDER BY d0""").df()
first_day.to_csv(O / "02_first_seen_day.csv", index=False)

con.execute(f"""COPY (
  WITH f AS (SELECT userId, min(dt) AS d0 FROM imp GROUP BY userId HAVING min(dt) <= {LAST_D0})
  SELECT f.userId, f.d0,
    sum(CASE WHEN i.dt < f.d0 + {OBS} THEN 1 ELSE 0 END) AS obs_impr,
    sum(CASE WHEN i.dt < f.d0 + {OBS} THEN i.isClick ELSE 0 END) AS obs_clicks,
    count(DISTINCT CASE WHEN i.dt < f.d0 + {OBS} THEN i.dt END) AS obs_days,
    sum(CASE WHEN i.dt >= f.d0 + {OBS} AND i.dt < f.d0 + {OBS + OUT_LEN} THEN 1 ELSE 0 END) AS out_impr,
    sum(CASE WHEN i.dt >= f.d0 + {OBS} AND i.dt < f.d0 + {OBS + OUT_LEN} THEN i.isClick ELSE 0 END) AS out_clicks,
    count(DISTINCT CASE WHEN i.dt >= f.d0 + {OBS} AND i.dt < f.d0 + {OBS + OUT_LEN} AND i.isClick = 1 THEN i.dt END) AS out_click_days
  FROM f JOIN imp i USING (userId) GROUP BY f.userId, f.d0
) TO '{P}/user_windows.parquet' (FORMAT parquet)""")

u = con.execute(f"""SELECT w.*, TRY_CAST(u.registeredMonthCnt AS INT) AS tenure
  FROM '{P}/user_windows.parquet' w LEFT JOIN users u USING (userId)""").df()
con.close()

u["returned"] = (u.out_impr > 0).astype(int)
u["obs_ctr"] = u.obs_clicks / u.obs_impr
ret = u[u.returned == 1].copy()                     # users shown at least one card in the outcome window
ret["any_click"] = (ret.out_clicks > 0).astype(int)

In [10]:
# Cards shown in the outcome window, and single-measure baselines (ROC-AUC and PR-AUC)
labels = ["0", "1-4", "5-9", "10-19", "20-49", "50+"]
expo_bin = pd.cut(u.out_impr, bins=[-1, 1, 5, 10, 20, 50, 10**9], labels=labels, right=False)
expo = expo_bin.value_counts().reindex(labels)
any_click = (u.out_clicks > 0).groupby(expo_bin, observed=False).mean().reindex(labels)   # share who click at least once
(expo.rename("users").to_frame().assign(share=lambda d: d.users / d.users.sum(), share_any_click=any_click)
     .to_csv(O / "02_outcome_exposure.csv"))


def auc_pr(y, s):
    return dict(roc_auc=round(roc_auc_score(y, s), 4),
                pr_auc=round(average_precision_score(y, s), 4),
                base_rate=round(float(np.mean(y)), 4))


res = {
    "users_total": int(first_day.users.sum()),
    "main_cohort_users": int(len(u)),
    "main_cohort_share": round(len(u) / first_day.users.sum(), 4),
    "new_user_cohort_users": int((u.tenure <= 1).sum()),
    "returned_share": round(u.returned.mean(), 4),
    "inactive_share_among_returned_zero_clicks": round(1 - ret.any_click.mean(), 4),
    "returned_with_lt10_outcome_impr_share": round((ret.out_impr < 10).mean(), 4),
    "baselines_T1_return": {
        "obs_impressions": auc_pr(u.returned, u.obs_impr),
        "obs_visit_days": auc_pr(u.returned, u.obs_days),
        "obs_clicks": auc_pr(u.returned, u.obs_clicks)},
    "baselines_T2_any_click_given_return": {
        "obs_impressions": auc_pr(ret.any_click, ret.obs_impr),
        "obs_ctr": auc_pr(ret.any_click, ret.obs_ctr),
        "obs_clicks": auc_pr(ret.any_click, ret.obs_clicks),
        "outcome_impressions_(not_a_feature)": auc_pr(ret.any_click, ret.out_impr)},
}
new_users = ret[ret.tenure <= 1]
res["new_users_returned"] = int(len(new_users))
res["new_users_T2_obs_ctr"] = (auc_pr(new_users.any_click, new_users.obs_ctr)
                               if new_users.any_click.nunique() == 2 else None)
(O / "02_cohorts_baselines.json").write_text(json.dumps(res, indent=2))
pd.Series({k: v for k, v in res.items() if not isinstance(v, dict)}, name="value").to_frame()

,value
users_total,2.085602e+06
main_cohort_users,1.084258e+06
main_cohort_share,5.199000e-01
new_user_cohort_users,4.318500e+04
returned_share,5.071000e-01
inactive_share_among_returned_zero_clicks,7.604000e-01
returned_with_lt10_outcome_impr_share,4.585000e-01
new_users_returned,1.618700e+04


In [11]:
# Share who click at least once by the exact number of cards shown in the outcome window (100 or more pooled at 100),
# against what a constant click rate per card would give: 1 - (1 - p)^n, with p the cohort's click rate in the window
shown = u[u.out_impr > 0].assign(n=lambda d: d.out_impr.clip(upper=100), clicked=lambda d: (d.out_clicks > 0).astype(float))
curve = shown.groupby("n").agg(users=("clicked", "size"), share_any_click=("clicked", "mean"))
p_card = float(u.out_clicks.sum() / u.out_impr.sum())
curve["constant_rate_reference"] = 1 - (1 - p_card) ** curve.index.values
curve["click_rate_per_card"] = p_card
curve.round(5).to_csv(O / "02_outcome_exposure_curve.csv")
print(f"Click rate per card in the outcome window: {p_card:.4f}")
curve.loc[[1, 2, 5, 10, 20, 50, 100]].round(3)

Click rate per card in the outcome window: 0.0496


,users,share_any_click,constant_rate_reference,click_rate_per_card
n,,,,
1.0,104,0.000,0.050,0.05
2.0,259,0.015,0.097,0.05
5.0,12017,0.208,0.225,0.05
10.0,7660,0.270,0.399,0.05
20.0,11011,0.189,0.639,0.05
50.0,1538,0.565,0.922,0.05
100.0,36848,0.825,0.994,0.05


In [12]:
# Baselines: return (T1) and any click among users who returned (T2)
pd.concat({t: pd.DataFrame(res[t]).T for t in ("baselines_T1_return", "baselines_T2_any_click_given_return")})

roc_auc  \
baselines_T1_return                 obs_impressions                       0.6708   
                                    obs_visit_days                        0.6876   
                                    obs_clicks                            0.5426   
baselines_T2_any_click_given_return obs_impressions                       0.6755   
                                    obs_ctr                               0.6467   
                                    obs_clicks                            0.6590   
                                    outcome_impressions_(not_a_feature)   0.8384   

                                                                         pr_auc  \
baselines_T1_return                 obs_impressions                      0.6747   
                                    obs_visit_days                       0.6815   
                                    obs_clicks                           0.5474   
baselines_T2_any_click_given_return obs_impressions                      0.4570   
                                    obs_ctr                              0.3431   
                                    obs_clicks                           0.4265   
                                    outcome_impressions_(not_a_feature)  0.6514   

                                                                         base_rate  
baselines_T1_return                 obs_impressions                         0.5071  
                                    obs_visit_days                          0.5071  
                                    obs_clicks                              0.5071  
baselines_T2_any_click_given_return obs_impressions                         0.2396  
                                    obs_ctr                                 0.2396  
                                    obs_clicks                              0.2396  
                                    outcome_impressions_(not_a_feature)     0.2396

In [13]:
# Users by number of cards shown in the outcome window, and the share of them who click at least once
pd.read_csv(O / "02_outcome_exposure.csv", index_col=0).round(4)

,users,share,share_any_click
out_impr,,,
0,534409,0.4929,0.0000
1-4,142806,0.1317,0.0208
5-9,109317,0.1008,0.1031
10-19,110074,0.1015,0.1949
20-49,106742,0.0984,0.3694
50+,80910,0.0746,0.6996


In [14]:
# Users by first day seen in the log, and how many of them hold a new account (at most one month old on 1 December)
first_day.assign(new_account_share=lambda d: (d.users_new_account / d.users).round(4)).set_index("d0").T

d0,1,2,3,4,5,6,7,8,9,10,...,21,22,23,24,25,26,27,28,29,30
users,195904.0000,164002.0000,135590.0000,98358.0000,88953.0000,82185.000,76678.0000,78417.0000,87928.000,76243.0000,...,44327.0000,47875.0000,57525.0000,52025.0000,39113.0000,37305.000,35584.0000,36144.0000,41072.0,51196.0000
users_new_account,5866.0000,6145.0000,5108.0000,3521.0000,3339.0000,3123.000,3271.0000,3660.0000,5008.000,4144.0000,...,2994.0000,3708.0000,5541.0000,4568.0000,3329.0000,3060.000,2964.0000,2871.0000,4108.0,6406.0000
users_account_age_missing,7.0000,20.0000,13.0000,5.0000,4.0000,6.000,4.0000,10.0000,13.000,17.0000,...,22.0000,31.0000,26.0000,33.0000,17.0000,7.000,5.0000,5.0000,13.0,18.0000
new_account_share,0.0299,0.0375,0.0377,0.0358,0.0375,0.038,0.0427,0.0467,0.057,0.0544,...,0.0675,0.0775,0.0963,0.0878,0.0851,0.082,0.0833,0.0794,0.1,0.1251


This group holds 1,084,258 users, 52% of all users. Of these, 49% were shown no card in the two weeks, and 76% of those who returned clicked nothing.

The share who click at least once rises with the cards shown, from 2% of users shown one to four cards to 70% of those shown 50 or more. The number of cards shown in the two weeks ranks returning users by whether they click with an AUC of 0.84, against 0.68 for the best measure from the first week. Clicking at least once therefore mostly measures exposure, so the analysis predicts two outcomes: whether a user returns, and the share of the cards shown that the user clicks.

Only 5.9% of users first seen on days 2 to 30 hold an account a month old or younger. Most users first seen late in the month are existing users returning after a pause.

## 6. Time between visits

A visit day is a day on which the user is shown at least one card. The gaps between a user's consecutive visit days show how long the outcome period must be to capture most returns.

In [15]:
# Gaps between consecutive visit days of the same user
con = connect(keep_order=False)
con.execute(f"CREATE TABLE days AS SELECT DISTINCT userId, dt FROM read_parquet('{P}/impressions_place/d*.parquet')")
con.execute("""CREATE TABLE gaps AS SELECT dt - lag(dt) OVER (PARTITION BY userId ORDER BY dt) AS gap
               FROM days""")
dist = con.execute("SELECT gap, count(*) AS n FROM gaps WHERE gap IS NOT NULL GROUP BY gap ORDER BY gap").df()
dist["cum_share"] = dist.n.cumsum() / dist.n.sum()
dist.to_csv(O / "03_visit_gaps.csv", index=False)
per_user = con.execute("SELECT count(*) AS visit_days FROM days GROUP BY userId").df()

# Time to the next visit, allowing for the end of the month: every visit day starts a spell that ends at the next visit
# day (an event) or is cut off by day 30 (censored after 30 - dt days). Spells starting on day 30 have no time left.
spells = con.execute("""SELECT d, event, count(*) AS n FROM (
    SELECT coalesce(lead(dt) OVER w - dt, 30 - dt) AS d, (lead(dt) OVER w IS NOT NULL)::INT AS event
    FROM days WINDOW w AS (PARTITION BY userId ORDER BY dt))
  WHERE d > 0 GROUP BY d, event ORDER BY d, event""").df()

# How concentrated use is: cards shown per user
impr_user = con.execute(f"""SELECT count(*) AS impr FROM read_parquet('{P}/impressions_place/d*.parquet')
                            GROUP BY userId""").df().impr.sort_values(ascending=False).values
con.close()

# Kaplan-Meier estimate of the share of spells that have ended in a return by day d
ev = (spells.pivot_table(index="d", columns="event", values="n", aggfunc="sum", fill_value=0)
        .reindex(index=range(1, 30), columns=[0, 1], fill_value=0).rename(columns={0: "censored", 1: "events"}))
km = pd.DataFrame({"d": ev.index, "events": ev.events.values, "censored": ev.censored.values})
km["at_risk"] = (km.events + km.censored)[::-1].cumsum()[::-1].values      # spells still open at the start of day d
km["share_returned_km"] = 1 - np.cumprod(1 - km.events / km.at_risk)
km = km.merge(dist[["gap", "cum_share"]].rename(columns={"gap": "d", "cum_share": "share_returned_naive"}), on="d", how="left")
km.round(6).to_csv(O / "03_time_to_next_visit.csv", index=False)

# Visit days per user, and the share of all cards shown that goes to the top x% of users
per_user.visit_days.value_counts().sort_index().rename_axis("visit_days").rename("users").to_csv(O / "03_visit_days_per_user.csv")
cum = np.cumsum(impr_user) / impr_user.sum()
top = np.arange(0, 101) / 100
pd.DataFrame({"top_user_share": top,
              "card_share": np.r_[0, cum[np.maximum(np.ceil(top[1:] * len(cum)).astype(int) - 1, 0)]]}
             ).round(6).to_csv(O / "03_concentration.csv", index=False)

res = {
    "gaps_total": int(dist.n.sum()),
    "share_gap_le_7": round(float(dist.loc[dist.gap <= 7, "n"].sum() / dist.n.sum()), 4),
    "share_gap_le_14": round(float(dist.loc[dist.gap <= 14, "n"].sum() / dist.n.sum()), 4),
    "median_gap": int(dist.loc[dist.cum_share >= 0.5, "gap"].iloc[0]),
    "p90_gap": int(dist.loc[dist.cum_share >= 0.9, "gap"].iloc[0]),
    "users_one_visit_day_share": round(float((per_user.visit_days == 1).mean()), 4),
    "visit_days_median": float(per_user.visit_days.median()),
    "km_share_returned_within_7": round(float(km.set_index("d").share_returned_km[7]), 4),
    "km_share_returned_within_14": round(float(km.set_index("d").share_returned_km[14]), 4),
    "km_share_returned_within_29": round(float(km.set_index("d").share_returned_km[29]), 4),
    "spells_censored_share": round(float(km.censored.sum() / (km.censored.sum() + km.events.sum())), 4),
    "cards_share_top_1pct_users": round(float(cum[int(np.ceil(0.01 * len(cum))) - 1]), 4),
    "cards_share_top_10pct_users": round(float(cum[int(np.ceil(0.10 * len(cum))) - 1]), 4),
    "cards_share_top_50pct_users": round(float(cum[int(np.ceil(0.50 * len(cum))) - 1]), 4),
}
(O / "03_visit_gaps.json").write_text(json.dumps(res, indent=2))
pd.Series(res, name="value").to_frame()

,value
gaps_total,3.511871e+06
share_gap_le_7,8.312000e-01
share_gap_le_14,9.493000e-01
median_gap,2.000000e+00
p90_gap,1.100000e+01
users_one_visit_day_share,5.154000e-01
visit_days_median,1.000000e+00
km_share_returned_within_7,5.714000e-01
km_share_returned_within_14,6.859000e-01
km_share_returned_within_29,7.924000e-01


In [16]:
# The first 15 gap lengths and their cumulative share, completed gaps only and allowing for the end of the month
km.head(15)

,d,events,censored,at_risk,share_returned_km,share_returned_naive
0,1,1320507,137969,5383323,0.245296,0.376012
1,2,556266,111683,3924847,0.352260,0.534408
2,3,334270,93718,3256898,0.418740,0.629591
3,4,234387,87905,2828910,0.466900,0.696333
4,5,181874,82747,2506618,0.505580,0.748121
5,6,158159,99800,2241997,0.540459,0.793156
6,7,133644,97498,1984038,0.571413,0.831211
7,8,95478,76447,1752896,0.594758,0.858399
8,9,72848,66278,1580971,0.613430,0.879142
9,10,60396,61180,1441845,0.629623,0.896340


Across 3,511,871 gaps the median is 2 days, and 83% last 7 days or less. These shares count completed gaps only, and a gap that starts late in the month is seen only if it is short. Following every visit until the next one or the end of the data (Kaplan-Meier) gives a 57% chance of another visit within 7 days and 79% within 29 days. A 7-day period captures 72% of the returns seen within 29 days and leaves 23 days from which to predict.

Use is concentrated. Half of all users (51.5%) visit on a single day, and the 10% of users shown the most cards receive 64% of all cards shown.

## 7. Daily activity and click rate by place in the feed

Two descriptive series for the figures: users, cards shown and click rate on each day, and the click rate by the card's place in the feed, for all users and by how far users scroll.

In [17]:
# Users, impressions and click rate by day
con = connect(keep_order=False)
daily = con.execute(f"""SELECT CAST(dt AS INT) AS dt, count(DISTINCT userId) AS users, count(*) AS impressions,
    sum(isClick) AS clicks FROM read_parquet('{P}/impressions_place/d*.parquet') GROUP BY 1 ORDER BY 1""").df()
daily["click_rate"] = daily.clicks / daily.impressions
daily.to_csv(O / "02_daily_activity.csv", index=False)

# Click rate by position in the feed
pos = con.execute(f"""SELECT least(impressPosition, 51) AS position, count(*) AS impressions, sum(isClick) AS clicks
    FROM read_parquet('{P}/impressions_place/d*.parquet') GROUP BY 1 ORDER BY 1""").df()
con.close()
pos["click_rate"] = pos.clicks / pos.impressions
pos.to_csv(O / "02_click_rate_by_position.csv", index=False)

# The same by how far users usually scroll, from the share of their cards shown beyond the first page (place 4 or
# later) over the month: 1 = first page only, 2 = beyond it for under half of the cards, 3 = for half or more
con = connect(keep_order=False)
pos3 = con.execute(f"""WITH i AS (SELECT userId, impressPosition, isClick FROM read_parquet('{P}/impressions_place/d*.parquet')),
    g AS (SELECT userId, CASE WHEN avg((impressPosition >= 4)::INT) = 0 THEN 1
                              WHEN avg((impressPosition >= 4)::INT) < 0.5 THEN 2 ELSE 3 END AS scroll_group FROM i GROUP BY userId)
  SELECT scroll_group, least(impressPosition, 51) AS position, count(*) AS impressions, sum(isClick) AS clicks
  FROM i JOIN g USING (userId) GROUP BY 1, 2 ORDER BY 1, 2""").df()
con.close()
pos3["click_rate"] = pos3.clicks / pos3.impressions
pos3.to_csv(O / "02_click_rate_by_position_scroll.csv", index=False)
# users in each scroll-depth group (the same groups)
con = connect(keep_order=False)
groups = con.execute(f"""SELECT scroll_group, count(*) AS users FROM (
    SELECT userId, CASE WHEN avg((impressPosition >= 4)::INT) = 0 THEN 1
                        WHEN avg((impressPosition >= 4)::INT) < 0.5 THEN 2 ELSE 3 END AS scroll_group
    FROM read_parquet('{P}/impressions_place/d*.parquet') GROUP BY userId) GROUP BY 1 ORDER BY 1""").df()
con.close()
groups["share"] = groups.users / groups.users.sum()
groups.to_csv(O / "02_scroll_groups.csv", index=False)
daily.set_index("dt").T.round(4)

dt,1,2,3,4,5,6,7,8,9,10,...,21,22,23,24,25,26,27,28,29,30
users,1.959040e+05,2.139490e+05,2.110920e+05,1.756000e+05,1.758860e+05,1.764310e+05,1.768580e+05,1.864560e+05,2.081060e+05,1.953750e+05,...,1.762510e+05,1.916710e+05,2.136700e+05,205400.000,1.718700e+05,1.711270e+05,1.667400e+05,1.757960e+05,1.862370e+05,2.141500e+05
impressions,1.750647e+06,2.209482e+06,2.117063e+06,1.575821e+06,1.794558e+06,1.851646e+06,1.896851e+06,1.955904e+06,2.324319e+06,2.136265e+06,...,1.776319e+06,1.712245e+06,2.056344e+06,1953708.000,1.530434e+06,1.741718e+06,1.656995e+06,1.873255e+06,2.049953e+06,2.288097e+06
clicks,9.290300e+04,1.309690e+05,1.183000e+05,7.768600e+04,8.631800e+04,8.683800e+04,9.190900e+04,9.571400e+04,1.267480e+05,1.100040e+05,...,8.209500e+04,8.749600e+04,1.211140e+05,107443.000,7.313000e+04,7.966200e+04,7.038200e+04,8.633200e+04,1.002660e+05,1.299620e+05
click_rate,5.310000e-02,5.930000e-02,5.590000e-02,4.930000e-02,4.810000e-02,4.690000e-02,4.850000e-02,4.890000e-02,5.450000e-02,5.150000e-02,...,4.620000e-02,5.110000e-02,5.890000e-02,0.055,4.780000e-02,4.570000e-02,4.250000e-02,4.610000e-02,4.890000e-02,5.680000e-02


In [18]:
# The first 12 places in the feed, all users and by scroll-depth group (1 = first page only)
pos.head(12).set_index("position")[["impressions", "click_rate"]].join(
    pos3.pivot(index="position", columns="scroll_group", values="click_rate").add_prefix("click_rate_group_"))

,impressions,click_rate,click_rate_group_1,click_rate_group_2,click_rate_group_3
position,,,,,
0,9103137,0.054627,0.021373,0.051894,0.127977
1,9196595,0.020349,0.004916,0.018351,0.055277
2,9060285,0.026237,0.001976,0.023838,0.079969
3,6294941,0.021758,0.001110,0.019521,0.071073
4,713420,0.053705,NaN,0.026592,0.099124
5,1277574,0.044235,NaN,0.024917,0.074644
6,1037497,0.076287,NaN,0.051929,0.096208
7,643552,0.111217,NaN,0.078512,0.132029
8,704187,0.089610,NaN,0.063102,0.104720


Between 162,987 and 214,150 users see at least one card each day. The daily click rate moves between 4.2% and 5.9% and peaks at weekends, and 4.97% of all cards shown are clicked.

The four cards of the first page receive 58% of impressions but are clicked least, between 2.0% and 5.5% of the time, against 8% to 9% further down. Part of this is who is looking: 63% of users see first pages only, 26% go beyond the first page for less than half of their cards and 11% for half or more, and the last group clicks more at every place. The pattern remains within each group, which is why the click model compares cards shown to the same user at the same moment.

## 8. The snapshot panel

The panel has one row for each user and each day from 1 to 23 on which the user was shown at least one card. Features use the days up to and including that day, over the last 1, 3, 7 and 14 days and the whole history. Outcomes use the seven days after it, which is why day 23 is the last day scored.

Each impression carries its card's click rate on the whole platform up to the day before, pulled toward the platform average. A new session starts after more than 30 minutes without a card shown. Non-integer values are summed as integers so that the panel is identical on every run. `THESIS_SAMPLE=10` keeps 10% of users for quick tests.

In [19]:
# Working database, sampled users and the card click rates known on each day
PANEL = P / f"panel{TAG}.parquet"
DB = P / "_panel.duckdb"
DB.unlink(missing_ok=True)                                   # start from an empty working database
con = connect(DB, keep_order=False)
con.create_function("sample_bucket",
                    lambda s: int(hashlib.md5(("s" + s).encode()).hexdigest()[:8], 16) % 100,
                    ["VARCHAR"], "INTEGER")
con.execute(f"""CREATE TABLE IF NOT EXISTS users AS SELECT DISTINCT userId FROM read_parquet('{P}/impressions_place/d*.parquet')
                WHERE {SAMPLE} = 100 OR sample_bucket(userId) < {SAMPLE}""")
con.execute(f"CREATE TABLE IF NOT EXISTS card_prior AS {card_prior_sql(P)}")   # shared with the click model
g = platform_rate(con, P)
print(f"Platform click rate used as the prior: {g:.5f}")

Platform click rate used as the prior: 0.04952


### 8.1 One row per user and visit day

In [20]:
# Daily totals per user: cards shown, actions, viewing time, video share, position, expected clicks, sessions
con.execute(f"""CREATE TABLE IF NOT EXISTS daily AS
  WITH i AS (SELECT i.* FROM read_parquet('{P}/impressions_place/d*.parquet') i JOIN users USING (userId)),
  ic AS (SELECT i.userId, i.dt, i.impressTime, i.isClick, i.isLike, i.isShare, i.isComment, i.isViewComment,
                i.isIntoPersonalHomepage, i.mlogViewTime, i.impressPosition, m.type AS card_type,
                coalesce((cp.cum_clicks + {K} * {g}) / (cp.cum_impr + {K}), {g}) AS card_ctr,
                ln(1 + coalesce(cp.cum_impr, 0)) AS card_pop
         FROM i LEFT JOIN '{P}/mlog_demographics.parquet' m USING (mlogId)
         ASOF LEFT JOIN card_prior cp ON cp.mlogId = i.mlogId AND i.dt >= cp.dt_apply),
  s AS (SELECT userId, dt, sum(new) AS sessions FROM (
          SELECT userId, dt, CASE WHEN impressTime - lag(impressTime) OVER (PARTITION BY userId ORDER BY impressTime) > 1800000
                 OR lag(impressTime) OVER (PARTITION BY userId ORDER BY impressTime) IS NULL THEN 1 ELSE 0 END AS new FROM ic)
        GROUP BY userId, dt)
  SELECT ic.userId, ic.dt, count(*) AS impr, sum(isClick) AS clicks, sum(isLike) AS likes, sum(isShare) AS shares,
    sum(isComment) AS comments, sum(isViewComment) AS viewcomments, sum(isIntoPersonalHomepage) AS homepage,
    -- non-integer values are summed as integers (exact in any order); win() below divides by SCALE
    sum(CASE WHEN isClick = 1 THEN round(coalesce(mlogViewTime, 0) * 1e6)::BIGINT ELSE 0 END) AS viewtime,
    sum(CASE WHEN card_type = '2' THEN 1 ELSE 0 END) AS video_seen, sum(impressPosition) AS pos_sum,
    sum(round(card_ctr * 1e9)::BIGINT) AS expected_clicks, sum(round(card_pop * 1e9)::BIGINT) AS pop_sum,
    any_value(s.sessions) AS sessions
  FROM ic JOIN s USING (userId, dt) GROUP BY ic.userId, ic.dt""")

### 8.2 Features and outcomes of each snapshot

In [21]:
# Window sums ending at each origin (features) and the 7 days after it (outcomes)
cols = ["impr", "clicks", "likes", "shares", "comments", "viewcomments", "homepage", "viewtime",
        "video_seen", "pos_sum", "expected_clicks", "pop_sum", "sessions"]
SCALE = {"viewtime": "1e6", "expected_clicks": "1e9", "pop_sum": "1e9"}   # integer units of the non-integer sums


def win(c, frame):
    """SQL for the sum of column c over a window of days; integer-coded columns are scaled back."""
    s = f"sum({c}) OVER (PARTITION BY userId ORDER BY dt RANGE BETWEEN {frame})"
    return f"{s} / {SCALE[c]}" if c in SCALE else s


feat = []
for w in (1, 3, 7, 14):
    frame = f"{w - 1} PRECEDING AND CURRENT ROW"
    for c in ["impr", "clicks", "likes", "shares", "viewcomments", "viewtime", "sessions", "expected_clicks"]:
        feat.append(f"{win(c, frame)} AS w{w}_{c}")
    feat.append(f"count(*) OVER (PARTITION BY userId ORDER BY dt RANGE BETWEEN {frame}) AS w{w}_visit_days")
allh = "UNBOUNDED PRECEDING AND CURRENT ROW"
for c in cols:
    feat.append(f"{win(c, allh)} AS h_{c}")
feat.append(f"count(*) OVER (PARTITION BY userId ORDER BY dt RANGE BETWEEN {allh}) AS h_visit_days")
fut = "1 FOLLOWING AND 7 FOLLOWING"

con.execute(f"""CREATE OR REPLACE TABLE panel AS SELECT * FROM (
  SELECT userId, dt AS origin_day,
    dt - min(dt) OVER (PARTITION BY userId) + 1 AS history_days,
    dt - lag(dt) OVER (PARTITION BY userId ORDER BY dt) AS gap_since_prev_visit,
    {", ".join(feat)},
    coalesce(count(*) OVER (PARTITION BY userId ORDER BY dt RANGE BETWEEN {fut}), 0) AS y_visit_days,
    coalesce({win("impr", fut)}, 0) AS y_impr,
    coalesce({win("clicks", fut)}, 0) AS y_clicks
  FROM daily) WHERE origin_day <= 23""")
con.execute(f"""COPY (SELECT p.*, TRY_CAST(u.registeredMonthCnt AS INT) AS tenure, dayofweek(DATE '2019-10-31' + p.origin_day::INT) AS origin_weekday
  FROM panel p LEFT JOIN '{P}/user_demographics.parquet' u USING (userId) ORDER BY p.userId, p.origin_day)
  TO '{PANEL}' (FORMAT parquet)""")
con.close()
DB.unlink()

In [22]:
# Panel size and outcome rates
duckdb.connect().execute(f"""SELECT count(*) AS rows, count(DISTINCT userId) AS users, avg(history_days) AS mean_history_days,
  avg((y_visit_days > 0)::INT) AS return7, sum(y_clicks)::DOUBLE / nullif(sum(y_impr), 0) AS out_ctr FROM '{PANEL}'""").df()

,rows,users,mean_history_days,return7,out_ctr
0,4306153,1793163,6.54375,0.570291,0.050506


The panel has 4,306,153 snapshots of 1,793,163 users. In 57.0% of them the user returns within the next seven days, and 5.05% of the cards shown in those days are clicked.

## 9. Summary

The data can be used almost as released once the feed position is converted to one scale, and no row is deleted. Age and gender depend on the kind of account and are used only for description. Clicking at least once mostly measures exposure, so return and clicks per card shown are predicted separately. Seven days capture most returns. The panel in `parquet/panel.parquet` is the input of every later notebook.